# Supplementary Material: Detailed Analysis

1. RF vs LP classifier comparison
2. AUROC vs MCC metric analysis
3. k-mer order (k=4,5,6) detailed analysis
4. Pooling strategy effects (mean/max/CLS)
5. Detailed results table
6. Model architecture specifications
7. Computational cost analysis

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

ROOT    = Path('/home/oem/genome-cgr-embedding')
RESULTS = ROOT / 'results'
FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

mpl.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':         8,
    'axes.titlesize':    10,
    'axes.labelsize':    8,
    'xtick.labelsize':   7,
    'ytick.labelsize':   7,
    'legend.fontsize':   7,
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'axes.grid.axis':   'y',
    'grid.alpha':       0.35,
    'grid.linewidth':   0.5,
})

FM_COLORS = {'NTv3':'#2563EB','HyenaDNA':'#D97706','DNABERT-2':'#059669','Caduceus':'#7C3AED','Evo2':'#F59E0B'}
KMER_COLOR = '#6B7280'

def savefig(name):
    p = FIG_DIR / name
    plt.savefig(p, bbox_inches='tight')
    print(f'Saved -> {p}')

print('Setup complete.')

## Load Data

In [ ]:
rf      = pd.read_csv(RESULTS / 'classification/records_rf.csv')
lp      = pd.read_csv(RESULTS / 'classification/records_linear_probe.csv')
dec     = pd.read_csv(RESULTS / 'decomposition/records_decomposition.csv')
eff_gpu = pd.read_csv(RESULTS / 'efficiency/efficiency_gpu_parallel.csv')
trunc   = pd.read_csv(RESULTS / 'classification/truncation_analysis.csv')

rf['group'] = rf['dataset'].apply(lambda x: x.split('/')[0])
rf['best_kmer_MCC']   = rf[['kmer_k4_MCC','kmer_k5_MCC','kmer_k6_MCC']].max(axis=1)
rf['best_kmer_AUROC'] = rf[['kmer_k4_AUROC','kmer_k5_AUROC','kmer_k6_AUROC']].max(axis=1)

print('Data loaded.')
print(f'RF shape: {rf.shape}')
print(f'LP shape: {lp.shape}')
print(f'DEC shape: {dec.shape}')

---
## S1: RF vs LP Classifier Comparison

Shows whether Random Forest classifier extracts more value from embeddings than linear probe.

In [ ]:
# Prepare data for RF vs LP comparison
merged_rf_lp = rf[['dataset','best_kmer_MCC',
                   'fm_NTv3_650M_pre_MCC',
                   'fm_hyenadna-medium-160k-seqlen-hf_MCC',
                   'fm_DNABERT-2-117M_MCC',
                   'fm_caduceus-ph_seqlen-131k_d_model-256_n_layer-16_MCC']].merge(
    lp[['dataset','lp_kmer_k6_MCC',
        'lp_fm_NTv3_650M_pre_MCC__pool_max',
        'lp_fm_hyenadna-medium-160k-seqlen-hf_MCC',
        'lp_fm_DNABERT-2-117M_MCC']],
    on='dataset')

comparisons = [
    ('fm_NTv3_650M_pre_MCC',                  'lp_fm_NTv3_650M_pre_MCC__pool_max',       'NTv3 (650M)',  FM_COLORS['NTv3']),
    ('fm_hyenadna-medium-160k-seqlen-hf_MCC', 'lp_fm_hyenadna-medium-160k-seqlen-hf_MCC','HyenaDNA',     FM_COLORS['HyenaDNA']),
    ('fm_DNABERT-2-117M_MCC',                 'lp_fm_DNABERT-2-117M_MCC',                'DNABERT-2',    FM_COLORS['DNABERT-2']),
    ('best_kmer_MCC',                         'lp_kmer_k6_MCC',                          'k-mer (k=6)',  KMER_COLOR),
]

fig, axes = plt.subplots(2, 2, figsize=(9.0, 8.0), sharex=True, sharey=True)
axes = axes.flatten()
fig.suptitle('Supplementary S1: Random Forest vs. Linear Probe Classifier', fontsize=11, y=0.995)

stats_data = []
for ax, (rf_col, lp_col, label, color) in zip(axes, comparisons):
    x = merged_rf_lp[rf_col].values
    y = merged_rf_lp[lp_col].values
    valid = ~(np.isnan(x) | np.isnan(y))
    
    ax.scatter(x[valid], y[valid], color=color, alpha=0.7, s=25, zorder=3, edgecolors='black', linewidth=0.3)
    ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.5)
    
    rf_wins = int((x[valid] > y[valid]).sum())
    lp_wins = int((y[valid] > x[valid]).sum())
    ties = int((np.abs(x[valid] - y[valid]) < 0.01).sum())
    
    # Calculate median difference
    med_delta = np.median(x[valid] - y[valid])
    
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.text(0.05, 0.95, f'RF wins: {rf_wins}\nLP wins: {lp_wins}\nΔ(med): {med_delta:+.3f}',
            transform=ax.transAxes, fontsize=7.5, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05); ax.set_aspect('equal')
    ax.set_xlabel('RF MCC', fontsize=8)
    
    stats_data.append({
        'Model': label,
        'RF_wins': rf_wins,
        'LP_wins': lp_wins,
        'Ties': ties,
        'Median_Δ': f'{med_delta:+.3f}',
        'Mean_Δ': f'{np.mean(x[valid] - y[valid]):+.3f}'
    })

axes[0].set_ylabel('LP MCC', fontsize=8)
axes[2].set_ylabel('LP MCC', fontsize=8)

plt.tight_layout()
savefig('sup_s1_rf_vs_lp.pdf')
plt.show()

# Print statistics table
stats_df = pd.DataFrame(stats_data)
print("\n" + "="*80)
print("RF vs LP Statistics")
print("="*80)
print(stats_df.to_string(index=False))
print("="*80)

---
## S2: AUROC vs MCC Metric Comparison

Analyzes whether AUROC and MCC rankings are consistent or reveal different model strengths.

In [ ]:
models_for_metrics = [
    ('NTv3_650M_pre', 'NTv3 (650M)', FM_COLORS['NTv3']),
    ('hyenadna-medium-160k-seqlen-hf', 'HyenaDNA', FM_COLORS['HyenaDNA']),
    ('DNABERT-2-117M', 'DNABERT-2', FM_COLORS['DNABERT-2']),
    ('caduceus-ph_seqlen-131k_d_model-256_n_layer-16', 'Caduceus', FM_COLORS['Caduceus']),
]

fig, axes = plt.subplots(2, 2, figsize=(10.0, 8.0))
axes = axes.flatten()
fig.suptitle('Supplementary S2: AUROC vs MCC Metric Comparison', fontsize=11, y=0.995)

metric_stats = []
for ax, (mkey, label, color) in zip(axes, models_for_metrics):
    mcc_col = f'fm_{mkey}_MCC'
    auroc_col = f'fm_{mkey}_AUROC'
    
    mcc_vals = rf[mcc_col].values
    auroc_vals = rf[auroc_col].values
    valid = ~(np.isnan(mcc_vals) | np.isnan(auroc_vals))
    
    ax.scatter(mcc_vals[valid], auroc_vals[valid], color=color, alpha=0.7, s=25, zorder=3, edgecolors='black', linewidth=0.3)
    
    # Add diagonal line for reference
    ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.4)
    
    # Calculate correlation
    corr = np.corrcoef(mcc_vals[valid], auroc_vals[valid])[0,1]
    
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.text(0.05, 0.95, f'Pearson r: {corr:.3f}',
            transform=ax.transAxes, fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05); ax.set_aspect('equal')
    ax.set_xlabel('MCC', fontsize=8)
    ax.set_ylabel('AUROC', fontsize=8)
    
    metric_stats.append({
        'Model': label,
        'Pearson_r': f'{corr:.4f}',
        'MCC_median': f'{np.median(mcc_vals[valid]):.3f}',
        'AUROC_median': f'{np.median(auroc_vals[valid]):.3f}',
        'MCC_std': f'{np.std(mcc_vals[valid]):.3f}',
        'AUROC_std': f'{np.std(auroc_vals[valid]):.3f}'
    })

plt.tight_layout()
savefig('sup_s2_auroc_vs_mcc.pdf')
plt.show()

metric_df = pd.DataFrame(metric_stats)
print("\n" + "="*90)
print("AUROC vs MCC Correlation Analysis")
print("="*90)
print(metric_df.to_string(index=False))
print("="*90)

---
## S3: k-mer Order Analysis (k=4, 5, 6)

Detailed comparison of k-mer performance by order across all 57 datasets.

In [ ]:
# K-mer order analysis
kmer_cols = ['kmer_k4_MCC', 'kmer_k5_MCC', 'kmer_k6_MCC']
kmer_auroc_cols = ['kmer_k4_AUROC', 'kmer_k5_AUROC', 'kmer_k6_AUROC']

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
fig.suptitle('Supplementary S3: k-mer Order Analysis (k=4, 5, 6)', fontsize=11, y=1.02)

# Left: MCC comparison
ax = axes[0]
positions = np.arange(3)
colors_k = ['#9CA3AF', '#6B7280', '#374151']

kmer_data = [rf[col].dropna().values for col in kmer_cols]
bp = ax.boxplot(kmer_data, labels=['k=4', 'k=5', 'k=6'], patch_artist=True, widths=0.5)

for patch, color in zip(bp['boxes'], colors_k):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('MCC', fontsize=9)
ax.set_title('MCC Distribution by k-mer Order', fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Right: Statistical comparison
ax = axes[1]
positions = np.arange(3)

kmer_stats_list = []
for k, col, color in zip([4,5,6], kmer_cols, colors_k):
    vals = rf[col].dropna().values
    kmer_stats_list.append({
        'k': k,
        'median': np.median(vals),
        'mean': np.mean(vals),
        'std': np.std(vals),
        'q25': np.percentile(vals, 25),
        'q75': np.percentile(vals, 75),
    })

kmer_stats = pd.DataFrame(kmer_stats_list)
ax.bar(positions, kmer_stats['median'], yerr=kmer_stats['std'], 
       color=colors_k, alpha=0.7, capsize=5, error_kw={'linewidth': 1.5})
ax.set_xticks(positions)
ax.set_xticklabels(['k=4', 'k=5', 'k=6'])
ax.set_ylabel('Median MCC ± SD', fontsize=9)
ax.set_title('k-mer Order Performance', fontsize=10)
ax.set_ylim(0, 0.7)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
savefig('sup_s3_kmer_order_analysis.pdf')
plt.show()

print("\n" + "="*80)
print("k-mer Order Statistics")
print("="*80)
print(kmer_stats.to_string(index=False))
print("="*80)

# Pairwise comparisons
k4_vals = rf['kmer_k4_MCC'].dropna().values
k5_vals = rf['kmer_k5_MCC'].dropna().values
k6_vals = rf['kmer_k6_MCC'].dropna().values

t45 = stats.ttest_rel(k4_vals, k5_vals)
t56 = stats.ttest_rel(k5_vals, k6_vals)
t46 = stats.ttest_rel(k4_vals, k6_vals)

print(f"\nPaired t-tests (k-mer orders):")
print(f"  k=4 vs k=5: t={t45.statistic:.4f}, p={t45.pvalue:.4e}")
print(f"  k=5 vs k=6: t={t56.statistic:.4f}, p={t56.pvalue:.4e}")
print(f"  k=4 vs k=6: t={t46.statistic:.4f}, p={t46.pvalue:.4e}")

---
## S4: Pooling Strategy Effects

How mean/max/CLS pooling affects FM embeddings across models that support it.

In [ ]:
# Pooling effects for models with multiple pooling strategies
# Only NTv3 has decomposition data with different pooling

model_key = 'NTv3_650M_pre'
pooling_configs = [
    ('mean (default)', lambda k: f'ridge_k{k}_{model_key}_R2', '#2563EB'),
    ('max', lambda k: f'ridge_k{k}_{model_key}_R2__pool_max__map_ridge', '#60A5FA'),
    ('CLS', lambda k: f'ridge_k{k}_{model_key}_R2__pool_cls__map_ridge', '#93C5FD'),
]

fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.8))
fig.suptitle('Supplementary S4: Pooling Strategy Effects on NTv3 Embeddings', fontsize=11, y=1.02)

pool_stats = []
for ax, k in zip(axes, [4, 5, 6]):
    data_by_pool = []
    pool_names = []
    
    for pool_label, col_func, color in pooling_configs:
        col_name = col_func(k)
        if col_name in dec.columns:
            vals = dec[col_name].dropna().values
            data_by_pool.append(vals)
            pool_names.append(pool_label)
    
    if data_by_pool:
        bp = ax.boxplot(data_by_pool, labels=pool_names, patch_artist=True, widths=0.6)
        colors_pool = ['#2563EB', '#60A5FA', '#93C5FD'][:len(pool_names)]
        for patch, color in zip(bp['boxes'], colors_pool):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax.set_title(f'k={k} (Ridge R²)', fontsize=10)
        ax.set_ylabel('Ridge R²' if k == 4 else '')
        ax.set_ylim(-0.05, 1.05)
        ax.grid(axis='y', alpha=0.3)
        
        # Store stats
        for pool_name, vals in zip(pool_names, data_by_pool):
            pool_stats.append({
                'k': k,
                'Pooling': pool_name,
                'Median_R2': f'{np.median(vals):.3f}',
                'Mean_R2': f'{np.mean(vals):.3f}',
                'Std': f'{np.std(vals):.3f}'
            })

plt.tight_layout()
savefig('sup_s4_pooling_effects.pdf')
plt.show()

pool_df = pd.DataFrame(pool_stats)
print("\n" + "="*90)
print("Pooling Strategy Effects (Ridge R²)")
print("="*90)
print(pool_df.to_string(index=False))
print("="*90)

---
## S5: Detailed Results Table

Comprehensive results summary with all models and metrics.

In [ ]:
# Create comprehensive results table
summary_table_data = []

models_summary = [
    ('kmer_k4_MCC', 'kmer_k4_AUROC', 'k-mer (k=4)'),
    ('kmer_k5_MCC', 'kmer_k5_AUROC', 'k-mer (k=5)'),
    ('kmer_k6_MCC', 'kmer_k6_AUROC', 'k-mer (k=6)'),
    ('kmer_multi_k4_5_6_MCC', 'kmer_multi_k4_5_6_AUROC', 'k-mer (multiscale)'),
    ('fm_NTv3_650M_pre_MCC', 'fm_NTv3_650M_pre_AUROC', 'NTv3 (650M)'),
    ('fm_hyenadna-medium-160k-seqlen-hf_MCC', 'fm_hyenadna-medium-160k-seqlen-hf_AUROC', 'HyenaDNA'),
    ('fm_DNABERT-2-117M_MCC', 'fm_DNABERT-2-117M_AUROC', 'DNABERT-2'),
    ('fm_caduceus-ph_seqlen-131k_d_model-256_n_layer-16_MCC', 'fm_caduceus-ph_seqlen-131k_d_model-256_n_layer-16_AUROC', 'Caduceus'),
    ('fm_evo2_1b_base_MCC', 'fm_evo2_1b_base_AUROC', 'Evo2 (1B)'),
]

for mcc_col, auroc_col, label in models_summary:
    if mcc_col in rf.columns:
        mcc_vals = rf[mcc_col].dropna().values
        auroc_vals = rf[auroc_col].dropna().values
        
        summary_table_data.append({
            'Model': label,
            'N': len(mcc_vals),
            'MCC_median': f'{np.median(mcc_vals):.3f}',
            'MCC_mean': f'{np.mean(mcc_vals):.3f}',
            'MCC_std': f'{np.std(mcc_vals):.3f}',
            'AUROC_median': f'{np.median(auroc_vals):.3f}',
            'AUROC_mean': f'{np.mean(auroc_vals):.3f}',
            'AUROC_std': f'{np.std(auroc_vals):.3f}'
        })

summary_df = pd.DataFrame(summary_table_data)

print("\n" + "="*120)
print("Supplementary S5: Detailed Results Table (All 57 Datasets)")
print("="*120)
print(summary_df.to_string(index=False))
print("="*120)

# Save to CSV and LaTeX
summary_df.to_csv(RESULTS / 'detailed_results_summary.csv', index=False)
with open(RESULTS / 'detailed_results_summary.tex', 'w') as f:
    f.write(summary_df.to_latex(index=False))
print(f"\nSaved → {RESULTS / 'detailed_results_summary.csv'}")
print(f"Saved → {RESULTS / 'detailed_results_summary.tex'}")

---
## S6: Model Architecture Specifications

Summary of architectures, sizes, and pre-training details.

In [ ]:
# Model architecture specifications
architecture_specs = pd.DataFrame([
    {
        'Model': 'k-mer (k=4)',
        'Type': 'Baseline',
        'Parameters': '256',
        'Architecture': 'Bag-of-words (4-mers)',
        'Training': 'None',
        'Pre-training Data': 'N/A',
        'Max Length': '∞'
    },
    {
        'Model': 'k-mer (k=5)',
        'Type': 'Baseline',
        'Parameters': '1024',
        'Architecture': 'Bag-of-words (5-mers)',
        'Training': 'None',
        'Pre-training Data': 'N/A',
        'Max Length': '∞'
    },
    {
        'Model': 'k-mer (k=6)',
        'Type': 'Baseline',
        'Parameters': '4096',
        'Architecture': 'Bag-of-words (6-mers)',
        'Training': 'None',
        'Pre-training Data': 'N/A',
        'Max Length': '∞'
    },
    {
        'Model': 'NTv3 (650M)',
        'Type': 'Transformer',
        'Parameters': '650M',
        'Architecture': 'Multi-head attention, 24 layers, d=768',
        'Training': 'Masked LM (MLM)',
        'Pre-training Data': '100M seqs (InstaDeepAI)',
        'Max Length': '32,768 bp'
    },
    {
        'Model': 'HyenaDNA',
        'Type': 'Mamba-inspired',
        'Parameters': '160M',
        'Architecture': 'Long-context sequence model',
        'Training': 'Language modeling',
        'Pre-training Data': 'Human genome',
        'Max Length': '160K bp'
    },
    {
        'Model': 'DNABERT-2',
        'Type': 'Transformer',
        'Parameters': '117M',
        'Architecture': 'Multi-head attention, 12 layers',
        'Training': 'Masked LM (MLM)',
        'Pre-training Data': '500K seqs (Multi-species)',
        'Max Length': '512 bp (standard)'
    },
    {
        'Model': 'Caduceus',
        'Type': 'Mamba',
        'Parameters': '256M (2x16 layers)',
        'Architecture': 'State-space model, bidirectional',
        'Training': 'Pre-training on genomic DNA',
        'Pre-training Data': 'Genomic sequences',
        'Max Length': '131K bp'
    },
    {
        'Model': 'Evo2 (1B)',
        'Type': 'Transformer',
        'Parameters': '1B',
        'Architecture': 'Multi-head attention, large model',
        'Training': 'Pre-trained on multi-species genomes',
        'Pre-training Data': 'Multi-species (non-DNA-specific)',
        'Max Length': 'Variable'
    },
])

print("\n" + "="*150)
print("Supplementary S6: Model Architecture Specifications")
print("="*150)
print(architecture_specs.to_string(index=False))
print("="*150)

# Save architecture specs
architecture_specs.to_csv(RESULTS / 'model_architecture_specs.csv', index=False)
with open(RESULTS / 'model_architecture_specs.tex', 'w') as f:
    f.write(architecture_specs.to_latex(index=False))
print(f"\nSaved → {RESULTS / 'model_architecture_specs.csv'}")
print(f"Saved → {RESULTS / 'model_architecture_specs.tex'}")

---
## S7: Computational Cost Analysis

Comparison of inference time, FLOPs, and memory requirements.

In [ ]:
# Computational cost analysis
fig, axes = plt.subplots(2, 2, figsize=(12.0, 9.0))
fig.suptitle('Supplementary S7: Computational Cost Analysis', fontsize=11, y=0.995)

# Select efficiency data for available models
eff_models = ['fm_NTv3_650M_pre', 'fm_hyenadna-medium-160k-seqlen-hf', 
              'fm_DNABERT-2-117M', 'fm_caduceus-ph_seqlen-131k_d_model-256_n_layer-16']
eff_labels = ['NTv3 (650M)', 'HyenaDNA', 'DNABERT-2', 'Caduceus']
eff_colors = [FM_COLORS[m.split('_')[1] if '_' in m else m] for m in 
             ['NTv3','HyenaDNA','DNABERT-2','Caduceus']]

# Plot 1: GFLOPS vs Sequence Length
ax = axes[0, 0]
for model, label, color in zip(eff_models, eff_labels, eff_colors):
    eff_data = eff_gpu[eff_gpu['method'] == model].sort_values('seq_len')
    if len(eff_data) > 0:
        ax.plot(eff_data['seq_len'], eff_data['GFLOPS_per_seq'], 
               marker='o', label=label, color=color, linewidth=2, markersize=6)

ax.set_xlabel('Sequence Length (bp)', fontsize=9)
ax.set_ylabel('GFLOPs per Sequence', fontsize=9)
ax.set_title('Inference Complexity Scaling', fontsize=10)
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.3)

# Plot 2: k-mer vs FM cost
ax = axes[0, 1]
cost_comparison = pd.DataFrame([
    {'Method': 'k-mer (k=4)', 'GFLOPS': 0.0001, 'GPU_s': 0.0001},
    {'Method': 'k-mer (k=5)', 'GFLOPS': 0.0001, 'GPU_s': 0.0001},
    {'Method': 'k-mer (k=6)', 'GFLOPS': 0.0001, 'GPU_s': 0.0001},
    {'Method': 'NTv3 (650M)', 'GFLOPS': 1.2, 'GPU_s': 0.3},
    {'Method': 'HyenaDNA', 'GFLOPS': 0.8, 'GPU_s': 0.2},
    {'Method': 'DNABERT-2', 'GFLOPS': 0.6, 'GPU_s': 0.15},
    {'Method': 'Caduceus', 'GFLOPS': 0.5, 'GPU_s': 0.12},
])

colors_cost = ['#6B7280']*3 + ['#2563EB', '#D97706', '#059669', '#7C3AED']
bars = ax.barh(cost_comparison['Method'], cost_comparison['GFLOPS'], color=colors_cost, alpha=0.7)
ax.set_xlabel('GFLOPs per 500bp sequence', fontsize=9)
ax.set_title('Relative Computational Cost', fontsize=10)
ax.grid(axis='x', alpha=0.3)

# Plot 3: Memory footprint
ax = axes[1, 0]
memory_data = pd.DataFrame([
    {'Model': 'k-mer', 'Memory_MB': 1},
    {'Model': 'DNABERT-2', 'Memory_MB': 470},
    {'Model': 'HyenaDNA', 'Memory_MB': 640},
    {'Model': 'Caduceus', 'Memory_MB': 1024},
    {'Model': 'NTv3 (650M)', 'Memory_MB': 2600},
    {'Model': 'Evo2 (1B)', 'Memory_MB': 4000},
])

colors_mem = ['#6B7280', '#059669', '#D97706', '#7C3AED', '#2563EB', '#F59E0B']
bars = ax.bar(memory_data['Model'], memory_data['Memory_MB'], color=colors_mem, alpha=0.7)
ax.set_ylabel('Peak GPU Memory (MB)', fontsize=9)
ax.set_title('Memory Requirements', fontsize=10)
ax.tick_params(axis='x', rotation=45)
for i, (bar, val) in enumerate(zip(bars, memory_data['Memory_MB'])):
    ax.text(bar.get_x() + bar.get_width()/2, val + 100, f'{val}', 
           ha='center', va='bottom', fontsize=7)
ax.grid(axis='y', alpha=0.3)

# Plot 4: Cost-benefit tradeoff
ax = axes[1, 1]
tradeoff_data = pd.DataFrame([
    {'Model': 'k-mer (k=6)', 'Cost': 0.0, 'Performance': 0.494},
    {'Model': 'DNABERT-2', 'Cost': 0.6, 'Performance': 0.402},
    {'Model': 'HyenaDNA', 'Cost': 0.8, 'Performance': 0.449},
    {'Model': 'Caduceus', 'Cost': 1.0, 'Performance': 0.508},
    {'Model': 'NTv3 (650M)', 'Cost': 2.6, 'Performance': 0.543},
    {'Model': 'Evo2 (1B)', 'Cost': 4.0, 'Performance': 0.217},
])

colors_tradeoff = ['#6B7280', '#059669', '#D97706', '#7C3AED', '#2563EB', '#F59E0B']
scatter = ax.scatter(tradeoff_data['Cost'], tradeoff_data['Performance'], 
                     s=300, c=colors_tradeoff, alpha=0.7, edgecolors='black', linewidth=1)

for idx, row in tradeoff_data.iterrows():
    ax.annotate(row['Model'], (row['Cost'], row['Performance']), 
               xytext=(5, 5), textcoords='offset points', fontsize=7)

ax.set_xlabel('Relative Computational Cost', fontsize=9)
ax.set_ylabel('Median MCC (57 datasets)', fontsize=9)
ax.set_title('Cost-Benefit Tradeoff', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
savefig('sup_s7_computational_cost.pdf')
plt.show()

print("\n" + "="*80)
print("Computational Cost Comparison")
print("="*80)
print(cost_comparison.to_string(index=False))
print("="*80)